In [1]:
import numpy as np
import os
import pandas as pd
import time
from scipy.spatial import cKDTree

from simulate import *
from model import *
from evaluation_utils import *

output_dir = "output/mcDETECT_output/"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir + "xtabs", exist_ok=True)

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Function for saving simulated raw data
def write_simulated_data(points_all, out_parquet, is_3d):
    df = points_all.copy().reset_index(drop = True)
    out = pd.DataFrame({"x": df["global_x"].astype(float),
                        "y": df["global_y"].astype(float),
                        "z": df["global_z"].astype(float) if is_3d else 0.0,
                        "gene": df["target"].astype(str)
                        })

    # Prefer explicit global granule identifier if present.
    if "granule_id" in df.columns:
        out["granule_id"] = df["granule_id"].astype(int)
    elif "id" in df.columns:
        out["granule_id"] = df["id"].astype(int)

    if "type" in df.columns:
        out["type"] = df["type"].astype(str)
    os.makedirs(os.path.dirname(out_parquet), exist_ok = True)
    out.to_parquet(out_parquet, index = False)

## Visualize sphere radius distribution

In [ ]:
point_type = ["CSR", "Extranuclear", "Intranuclear"]
ratio = [0.5, 0.25, 0.25]
mean_dist_extra = 1
mean_dist_intra = 4
beta_extra = (1, 19)
beta_intra = (19, 1)

simulate_z = True
name = "A"
density_overall = 0.16
num_clusters_extra = 10000
num_clusters_intra = 4000
seed = 1

for i in range(len(point_type)):
    simulate = simulation(name = name, density = density_overall * ratio[i], shape = (2000, 2000), layer_num = 8, layer_gap = 1.5, simulate_z = simulate_z, write_path = output_dir, seed = seed)
    if i == 0:
        points_CSR = simulate.simulate_CSR()
        points_CSR["type"] = [point_type[i]] * points_CSR.shape[0]
    elif i == 1:
        parents_cluster_extra, points_cluster_extra = simulate.simulate_cluster(num_clusters = num_clusters_extra, beta = beta_extra, mean_dist = mean_dist_extra)
        points_cluster_extra["type"] = [point_type[i]] * points_cluster_extra.shape[0]
    elif i == 2:
        parents_cluster_intra, points_cluster_intra = simulate.simulate_cluster(num_clusters = num_clusters_intra, beta = beta_intra, mean_dist = mean_dist_intra)
        points_cluster_intra["type"] = [point_type[i]] * points_cluster_intra.shape[0]
points_all = pd.concat([points_CSR, points_cluster_extra, points_cluster_intra], axis = 0, ignore_index = True)
parents_all = parents_cluster_extra

def rg(df):
    pts = df[["global_x", "global_y", "global_z"]].to_numpy()
    c = pts.mean(axis=0)
    return np.sqrt(((pts - c)**2).sum(axis=1).mean())

radii = points_cluster_intra.groupby("id").apply(rg)
med = np.nanmean(radii)
area = np.pi * (radii) ** 2
pd.DataFrame({"id": radii.index, "radius": radii, "area": area}).to_csv(output_dir + "intranuclear_area.csv", index = 0)

radii = points_cluster_extra.groupby("id").apply(rg)
med = np.nanmean(radii)
area = np.pi * (radii) ** 2
pd.DataFrame({"id": radii.index, "radius": radii, "area": area}).to_csv(output_dir + "extranuclear_area.csv", index = 0)

## Single-marker CSR and aggregation

In [ ]:
# Settings
point_type = ["CSR", "Extranuclear", "Intranuclear"]
ratio = [0.5, 0.25, 0.25]
mean_dist_extra = 1
mean_dist_intra = 4
beta_extra = (1, 19)
beta_intra = (19, 1)

In [ ]:
# Main simulation loop
dimension_settings = {"3D": True, "2D": False}

marker_settings = {"A": {"density": 0.08, "num_clusters_extra": 5000, "num_clusters_intra": 2000},
            "B": {"density": 0.04, "num_clusters_extra": 3000, "num_clusters_intra": 1200},
            "C": {"density": 0.02, "num_clusters_extra": 2000, "num_clusters_intra": 800}}

# marker_settings = {"D": {"density": 0.01, "num_clusters_extra": 1250, "num_clusters_intra": 500},
#                    "E": {"density": 0.005, "num_clusters_extra": 800, "num_clusters_intra": 300},
#                    "F": {"density": 0.0025, "num_clusters_extra": 500, "num_clusters_intra": 200}}

seed_lst = np.arange(1, 201)

for dimension, simulate_z in dimension_settings.items():
    
    if dimension == "2D":
        continue
    
    print(f"Running simulations for dimension: {dimension}")

    for name, params in marker_settings.items():
        
        print(f"Running simulations for marker: {name}")
        
        density_overall = params["density"]
        num_clusters_extra = params["num_clusters_extra"]
        num_clusters_intra = params["num_clusters_intra"]
        
        precision_lst = []
        recall_lst = []
        f1_lst = []
        time_lst = []

        for seed in seed_lst:

            # simulate data
            for i in range(len(point_type)):
                simulate = simulation(name = name, density = density_overall * ratio[i], shape = (2000, 2000), layer_num = 8, layer_gap = 1.5, simulate_z = simulate_z, write_path = output_dir, seed = seed)
                if i == 0:
                    points_CSR = simulate.simulate_CSR()
                    points_CSR["type"] = [point_type[i]] * points_CSR.shape[0]
                    points_CSR["id"] = -1
                elif i == 1:
                    parents_cluster_extra, points_cluster_extra = simulate.simulate_cluster(num_clusters = num_clusters_extra, beta = beta_extra, mean_dist = mean_dist_extra)
                    points_cluster_extra["type"] = [point_type[i]] * points_cluster_extra.shape[0]
                elif i == 2:
                    parents_cluster_intra, points_cluster_intra = simulate.simulate_cluster(num_clusters = num_clusters_intra, beta = beta_intra, mean_dist = mean_dist_intra)
                    points_cluster_intra["type"] = [point_type[i]] * points_cluster_intra.shape[0]
            points_all = pd.concat([points_CSR, points_cluster_extra, points_cluster_intra], axis = 0, ignore_index = True)
            parents_all = parents_cluster_extra
            
            # # save simulated and ground truth data
            # if dimension == "3D":
            #     write_simulated_data(points_all, f"simulated_data/single_marker/{dimension}/{name}/seed_{seed}.parquet", simulate_z)
            #     write_simulated_data(parents_all, f"simulated_data/single_marker/{dimension}/{name}/seed_{seed}_ground_truth.parquet", simulate_z)
            
            # points_all = pd.read_parquet(f"simulated_data/single_marker/{dimension}/{name}/seed_{seed}.parquet")
            
            # run mcDETECT
            detect = model(shape = (2000, 2000), transcripts = points_all, target_all = ["A", "B", "C"], eps = 1.5, in_thr = 0.25, size_thr = 4)
            # detect = model(shape = (2000, 2000), transcripts = points_all, target_all = ["D", "E", "F"], eps = 1.5, in_thr = 0.25, size_thr = 4)
            t0 = time.time()
            sphere = detect.dbscan_single(target_name = name)
            t1 = time.time()
            time_lst.append(t1 - t0)

            # new object-level metrics based on transcript labels
            metrics = compute_object_level_metrics(
                transcripts=points_all,
                spheres=sphere,
                tau_c=0.5,
                tau_p=0.5,
                type_col="type",
                granule_col="id",
                x_col="global_x",
                y_col="global_y",
                z_col="global_z",
                extranuclear_label="Extranuclear",
                return_crosstab=False
            )
            precision = metrics["precision"]
            recall = metrics["recall"]
            f1 = metrics["f1"]

            precision_lst.append(precision)
            recall_lst.append(recall)
            f1_lst.append(f1)
            
            if seed % 50 == 0:
                print(f"{seed} out of {len(seed_lst)} iterations!")
        
        time_df = pd.DataFrame({"Simulation": seed_lst.tolist(), "Time": time_lst})
        time_df.to_csv(output_dir + f"single_marker_{dimension}_{name}_{num_clusters_extra}_{num_clusters_intra}_time.csv", index = 0)

        results_df = pd.DataFrame({"Simulation": seed_lst.tolist(),
                                   "Precision": precision_lst,
                                   "Recall": recall_lst,
                                   "F1 Score": f1_lst})
        results_df.to_csv(output_dir + f"single_marker_{dimension}_{name}_{num_clusters_extra}_{num_clusters_intra}.csv", index = 0)

## Multi-marker CSR and aggregation

In [3]:
# Settings
name = ["A", "B", "C"]

shape = (2000, 2000)
layer_num = 8
layer_gap = 1.5
write_path = ""

CSR_density = [0.04, 0.02, 0.01]

extra_density = [0.02, 0.01, 0.005]
extra_num_clusters = 5000
extra_beta = (1, 19)
extra_comp_prob = [0.4, 0.3, 0.3]
extra_mean_dist = 1

intra_density = [0.02, 0.01, 0.005]
intra_num_clusters = 1000
intra_beta = (19, 1)
intra_comp_prob = [0.8, 0.1, 0.1]
intra_mean_dist = 4

In [ ]:
# Main simulation loop
dimension_settings = {"3D": True, "2D": False}
seed_lst = np.arange(1, 201)

for dimension, simulate_z in dimension_settings.items():
    
    if dimension == "2D":
        continue
    
    print(f"Running multi-marker simulations for dimension: {dimension}")

    precision_lst, recall_lst, f1_lst, time_lst = [], [], [], []

    for seed in seed_lst:
        
        # simulate data
        multi_simulate_extra = multi_simulation(name = name, density = extra_density, shape = shape, layer_num = layer_num, layer_gap = layer_gap, simulate_z = simulate_z, write_path = write_path, seed = seed)
        parents_extra, parents_all_extra, points_extra = multi_simulate_extra.simulate_cluster(num_clusters = extra_num_clusters, beta = extra_beta, comp_prob = extra_comp_prob, mean_dist = extra_mean_dist, comp_thr = 2)
        points_extra["type"] = ["Extranuclear"] * points_extra.shape[0]
        
        multi_simulate_intra = multi_simulation(name = name, density = intra_density, shape = shape, layer_num = layer_num, layer_gap = layer_gap, simulate_z = simulate_z, write_path = write_path, seed = seed + 100)
        parents_intra, parents_all_intra, points_intra = multi_simulate_intra.simulate_cluster(num_clusters = intra_num_clusters, beta = intra_beta, comp_prob = intra_comp_prob, mean_dist = intra_mean_dist, comp_thr = 2)
        points_intra["type"] = ["Intranuclear"] * points_intra.shape[0]
        
        simulate_A = simulation(name = name[0], density = CSR_density[0], shape = shape, layer_num = layer_num, layer_gap = layer_gap, simulate_z = simulate_z, write_path = write_path, seed = seed + 200)
        points_CSR_A = simulate_A.simulate_CSR()
        points_CSR_A["type"] = ["CSR"] * points_CSR_A.shape[0]
        points_CSR_A["granule_id"] = -1

        simulate_B = simulation(name = name[1], density = CSR_density[1], shape = shape, layer_num = layer_num, layer_gap = layer_gap, simulate_z = simulate_z, write_path = write_path, seed = seed + 300)
        points_CSR_B = simulate_B.simulate_CSR()
        points_CSR_B["type"] = ["CSR"] * points_CSR_B.shape[0]
        points_CSR_B["granule_id"] = -1

        simulate_C = simulation(name = name[2], density = CSR_density[2], shape = shape, layer_num = layer_num, layer_gap = layer_gap, simulate_z = simulate_z, write_path = write_path, seed = seed + 400)
        points_CSR_C = simulate_C.simulate_CSR()
        points_CSR_C["type"] = ["CSR"] * points_CSR_C.shape[0]
        points_CSR_C["granule_id"] = -1
        
        parents_all = parents_extra
        points_all = pd.concat([points_extra, points_intra, points_CSR_A, points_CSR_B, points_CSR_C], axis = 0, ignore_index = True)
        
        # check granule_id contains no NA
        if points_all["granule_id"].isna().any():
            raise ValueError("granule_id contains NA!")
        
        # # save simulated and ground truth data
        # if dimension == "3D":
        #     write_simulated_data(points_all, f"simulated_data/multi_marker/{dimension}/all/seed_{seed}.parquet", simulate_z)
        #     # parents_all from multi_simulation has id, global_* but no 'target'; add for write_simulated_data
        #     gt_df = parents_all.copy()
        #     if "target" not in gt_df.columns:
        #         gt_df["target"] = "gt"
        #     write_simulated_data(gt_df, f"simulated_data/multi_marker/{dimension}/all/seed_{seed}_ground_truth.parquet", simulate_z)
        
        # run mcDETECT on all genes
        detect_all = model(shape = (2000, 2000), transcripts = points_all, target_all = ["A", "B", "C"], eps = 1.5, in_thr = 0.25, comp_thr = 2, size_thr = 4, p = 0.2, l = 2.5)
        t0 = time.time()
        sphere_all = detect_all.merge_data()
        t1 = time.time()
        time_lst.append(t1 - t0)
        metrics, xtab = compute_object_level_metrics(
            transcripts=points_all,
            spheres=sphere_all,
            tau_c=0.5,
            tau_p=0.5,
            type_col="type",
            granule_col="granule_id",
            x_col="global_x",
            y_col="global_y",
            z_col="global_z",
            extranuclear_label="Extranuclear",
        )
        xtab.to_csv(output_dir + f"xtabs/multi_marker_{dimension}_all_{extra_num_clusters}_{intra_num_clusters}_seed_{seed}_xtab.csv", index = 0)
        precision = metrics["precision"]
        recall = metrics["recall"]
        f1 = metrics["f1"]
        precision_lst.append(precision)
        recall_lst.append(recall)
        f1_lst.append(f1)
        
        if seed % 50 == 0:
            print("{} out of {} iterations!".format(seed, len(seed_lst)))
    
    pd.DataFrame({"Simulation": seed_lst.tolist(), "Time": time_lst}).to_csv(output_dir + f"multi_marker_{dimension}_all_{extra_num_clusters}_{intra_num_clusters}_time.csv", index = 0)
    pd.DataFrame({"Simulation": seed_lst.tolist(), "Precision": precision_lst, "Recall": recall_lst, "F1": f1_lst}).to_csv(output_dir + f"multi_marker_{dimension}_all_{extra_num_clusters}_{intra_num_clusters}.csv", index = 0)

Running multi-marker simulations for dimension: 3D


## Benchmark ratio between CSR, extranuclear, and intranuclear aggregation

In [ ]:
# Benchmark 1: Vary CSR ratio, keep extra and intra-nuclear ratios identical
# Only use A, B, C markers and 3D case

# Original marker settings for A, B, C
marker_settings_original = {"A": {"density": 0.08, "num_clusters_extra": 5000, "num_clusters_intra": 2000},
                            "B": {"density": 0.04, "num_clusters_extra": 3000, "num_clusters_intra": 1200},
                            "C": {"density": 0.02, "num_clusters_extra": 2000, "num_clusters_intra": 800}}

# Original ratio
original_ratio = [0.5, 0.25, 0.25]  # [CSR, Extra, Intra]
original_extra_ratio = original_ratio[1]
original_intra_ratio = original_ratio[2]

# Test 5 different CSR ratios
csr_ratios = [0.2, 0.3, 0.4, 0.5, 0.6]

# Settings
point_type = ["CSR", "Extranuclear", "Intranuclear"]
mean_dist_extra = 1
mean_dist_intra = 4
beta_extra = (1, 19)
beta_intra = (19, 1)
simulate_z = True  # 3D only
shape_area = 2000 * 2000  # Used for scaling cluster counts

seed_lst = np.arange(1, 101)

print("Benchmark 1: Varying CSR ratio (keeping extra = intra)")
print(f"Testing CSR ratios: {csr_ratios}")

for csr_ratio in csr_ratios:
    # Calculate extra and intra ratios (they must be equal and sum with CSR to 1)
    remaining_ratio = 1.0 - csr_ratio
    extra_ratio = remaining_ratio / 2.0
    intra_ratio = remaining_ratio / 2.0
    ratio = [csr_ratio, extra_ratio, intra_ratio]
    
    print(f"\nCSR ratio: {csr_ratio:.2f}, Extra ratio: {extra_ratio:.2f}, Intra ratio: {intra_ratio:.2f}")
    
    for name, params_original in marker_settings_original.items():
        density_overall = params_original["density"]
        
        # Calculate scaling factors to maintain approximately same number of transcripts per cluster
        # Points per cluster ≈ (density * shape_area) / num_clusters
        # To keep points per cluster constant: num_clusters must scale proportionally with density change
        # num_clusters_new / num_clusters_original = new_density / original_density = new_ratio / original_ratio
        extra_scale_factor = extra_ratio / original_extra_ratio
        intra_scale_factor = intra_ratio / original_intra_ratio
        
        num_clusters_extra = int(params_original["num_clusters_extra"] * extra_scale_factor)
        num_clusters_intra = int(params_original["num_clusters_intra"] * intra_scale_factor)
        
        print(f"  Marker {name}: num_clusters_extra={num_clusters_extra}, num_clusters_intra={num_clusters_intra}")
        
        precision_lst = []
        recall_lst = []
        f1_lst = []
        
        for seed in seed_lst:
            # Simulate data
            for i in range(len(point_type)):
                simulate = simulation(name=name, density=density_overall * ratio[i], shape=(2000, 2000), 
                                     layer_num=8, layer_gap=1.5, simulate_z=simulate_z, 
                                     write_path=output_dir, seed=seed)
                if i == 0:
                    points_CSR = simulate.simulate_CSR()
                    points_CSR["type"] = [point_type[i]] * points_CSR.shape[0]
                    points_CSR["id"] = -1
                elif i == 1:
                    parents_cluster_extra, points_cluster_extra = simulate.simulate_cluster(
                        num_clusters=num_clusters_extra, beta=beta_extra, mean_dist=mean_dist_extra)
                    points_cluster_extra["type"] = [point_type[i]] * points_cluster_extra.shape[0]
                elif i == 2:
                    parents_cluster_intra, points_cluster_intra = simulate.simulate_cluster(
                        num_clusters=num_clusters_intra, beta=beta_intra, mean_dist=mean_dist_intra)
                    points_cluster_intra["type"] = [point_type[i]] * points_cluster_intra.shape[0]
            
            points_all = pd.concat([points_CSR, points_cluster_extra, points_cluster_intra], 
                                  axis=0, ignore_index=True)
            parents_all = parents_cluster_extra
            
            # Run mcDETECT
            detect = model(shape=(2000, 2000), transcripts=points_all, target_all=["A", "B", "C"], 
                          eps=1.5, in_thr=0.25, size_thr=4)
            sphere = detect.dbscan_single(target_name=name)
            
            # Legacy metrics (kept for reference)
            # tree = make_tree(d1=np.array(parents_all["global_x"]), 
            #                d2=np.array(parents_all["global_y"]), 
            #                d3=np.array(parents_all["global_z"]))
            # ground_truth_indices = set(parents_all.index)
            # precision, recall, accuracy, f1 = metric_main(tree, ground_truth_indices, sphere)

            # New object-level metrics
            metrics = compute_object_level_metrics(
                transcripts=points_all,
                spheres=sphere,
                tau_c=0.9,
                tau_p=0.9,
                type_col="type",
                granule_col="id",
                x_col="global_x",
                y_col="global_y",
                z_col="global_z",
                extranuclear_label="Extranuclear",
                return_crosstab=False
            )
            precision = metrics["precision"]
            recall = metrics["recall"]
            f1 = metrics["f1"]

            precision_lst.append(precision)
            recall_lst.append(recall)
            f1_lst.append(f1)
            
            if seed % 25 == 0:
                print(f"    {seed} out of {len(seed_lst)} iterations!")
        
        # Save results
        results_df = pd.DataFrame({
            "Simulation": seed_lst.tolist(),
            "Precision": precision_lst,
            "Recall": recall_lst,
            "F1_Score": f1_lst,
            "CSR_ratio": [csr_ratio] * len(seed_lst),
            "Extra_ratio": [extra_ratio] * len(seed_lst),
            "Intra_ratio": [intra_ratio] * len(seed_lst)
        })
        filename = f"benchmark_ratio_csr_{name}_csr{csr_ratio:.2f}_extra{extra_ratio:.2f}_intra{intra_ratio:.2f}.csv"
        results_df.to_csv(output_dir + filename, index=0)
        
        print(f"  Marker {name}: Mean Precision={np.mean(precision_lst):.4f}, "
              f"Recall={np.mean(recall_lst):.4f}, F1={np.mean(f1_lst):.4f}")

print("\nBenchmark 1 completed!")

In [ ]:
# Benchmark 2: Fix CSR at 0.5, vary the proportion of extra- and intra-nuclear ratios
# Only use A, B, C markers and 3D case

# Original marker settings for A, B, C
marker_settings_original = {"A": {"density": 0.08, "num_clusters_extra": 5000, "num_clusters_intra": 2000},
                            "B": {"density": 0.04, "num_clusters_extra": 3000, "num_clusters_intra": 1200},
                            "C": {"density": 0.02, "num_clusters_extra": 2000, "num_clusters_intra": 800}}

# Original ratio
original_ratio = [0.5, 0.25, 0.25]  # [CSR, Extra, Intra]
original_extra_ratio = original_ratio[1]
original_intra_ratio = original_ratio[2]

# Fix CSR at 0.5, test 5 different extra/intra splits
# Since CSR = 0.5, extra + intra must sum to 0.5
csr_ratio_fixed = 0.5
extra_intra_splits = [
    (0.4, 0.1),   # Extra-heavy
    (0.325, 0.175),
    (0.25, 0.25),  # Equal (original)
    (0.175, 0.325),
    (0.1, 0.4)    # Intra-heavy
]

# Settings
point_type = ["CSR", "Extranuclear", "Intranuclear"]
mean_dist_extra = 1
mean_dist_intra = 4
beta_extra = (1, 19)
beta_intra = (19, 1)
simulate_z = True  # 3D only
shape_area = 2000 * 2000  # Used for scaling cluster counts

seed_lst = np.arange(1, 101)

print("Benchmark 2: Varying extra/intra ratio (fixing CSR at 0.5)")
print(f"Testing extra/intra splits: {extra_intra_splits}")

for extra_ratio, intra_ratio in extra_intra_splits:
    ratio = [csr_ratio_fixed, extra_ratio, intra_ratio]
    
    print(f"\nCSR ratio: {csr_ratio_fixed:.2f}, Extra ratio: {extra_ratio:.2f}, Intra ratio: {intra_ratio:.2f}")
    
    for name, params_original in marker_settings_original.items():
        density_overall = params_original["density"]
        
        # Calculate scaling factors to maintain approximately same number of transcripts per cluster
        # Points per cluster ≈ (density * shape_area) / num_clusters
        # To keep points per cluster constant: num_clusters must scale proportionally with density change
        # num_clusters_new / num_clusters_original = new_density / original_density = new_ratio / original_ratio
        extra_scale_factor = extra_ratio / original_extra_ratio
        intra_scale_factor = intra_ratio / original_intra_ratio
        
        num_clusters_extra = int(params_original["num_clusters_extra"] * extra_scale_factor)
        num_clusters_intra = int(params_original["num_clusters_intra"] * intra_scale_factor)
        
        print(f"  Marker {name}: num_clusters_extra={num_clusters_extra}, num_clusters_intra={num_clusters_intra}")
        
        precision_lst = []
        recall_lst = []
        f1_lst = []
        
        for seed in seed_lst:
            # Simulate data
            for i in range(len(point_type)):
                simulate = simulation(name=name, density=density_overall * ratio[i], shape=(2000, 2000), 
                                     layer_num=8, layer_gap=1.5, simulate_z=simulate_z, 
                                     write_path=output_dir, seed=seed)
                if i == 0:
                    points_CSR = simulate.simulate_CSR()
                    points_CSR["type"] = [point_type[i]] * points_CSR.shape[0]
                    points_CSR["id"] = -1
                elif i == 1:
                    parents_cluster_extra, points_cluster_extra = simulate.simulate_cluster(
                        num_clusters=num_clusters_extra, beta=beta_extra, mean_dist=mean_dist_extra)
                    points_cluster_extra["type"] = [point_type[i]] * points_cluster_extra.shape[0]
                elif i == 2:
                    parents_cluster_intra, points_cluster_intra = simulate.simulate_cluster(
                        num_clusters=num_clusters_intra, beta=beta_intra, mean_dist=mean_dist_intra)
                    points_cluster_intra["type"] = [point_type[i]] * points_cluster_intra.shape[0]
            
            points_all = pd.concat([points_CSR, points_cluster_extra, points_cluster_intra], 
                                  axis=0, ignore_index=True)
            parents_all = parents_cluster_extra
            
            # Run mcDETECT
            detect = model(shape=(2000, 2000), transcripts=points_all, target_all=["A", "B", "C"], 
                          eps=1.5, in_thr=0.25, size_thr=4)
            sphere = detect.dbscan_single(target_name=name)
            
            # Legacy metrics (kept for reference)
            # tree = make_tree(d1=np.array(parents_all["global_x"]), 
            #                d2=np.array(parents_all["global_y"]), 
            #                d3=np.array(parents_all["global_z"]))
            # ground_truth_indices = set(parents_all.index)
            # precision, recall, accuracy, f1 = metric_main(tree, ground_truth_indices, sphere)

            # New object-level metrics
            metrics = compute_object_level_metrics(
                transcripts=points_all,
                spheres=sphere,
                tau_c=0.9,
                tau_p=0.9,
                type_col="type",
                granule_col="id",
                x_col="global_x",
                y_col="global_y",
                z_col="global_z",
                extranuclear_label="Extranuclear",
                return_crosstab=False
            )
            precision = metrics["precision"]
            recall = metrics["recall"]
            f1 = metrics["f1"]

            precision_lst.append(precision)
            recall_lst.append(recall)
            f1_lst.append(f1)
            
            if seed % 25 == 0:
                print(f"    {seed} out of {len(seed_lst)} iterations!")
        
        # Save results
        results_df = pd.DataFrame({
            "Simulation": seed_lst.tolist(),
            "Precision": precision_lst,
            "Recall": recall_lst,
            "F1_Score": f1_lst,
            "CSR_ratio": [csr_ratio_fixed] * len(seed_lst),
            "Extra_ratio": [extra_ratio] * len(seed_lst),
            "Intra_ratio": [intra_ratio] * len(seed_lst)
        })
        filename = f"benchmark_ratio_fixedcsr_{name}_csr{csr_ratio_fixed:.2f}_extra{extra_ratio:.3f}_intra{intra_ratio:.3f}.csv"
        results_df.to_csv(output_dir + filename, index=0)
        
        print(f"  Marker {name}: Mean Precision={np.mean(precision_lst):.4f}, "
              f"Recall={np.mean(recall_lst):.4f}, F1={np.mean(f1_lst):.4f}")

print("\nBenchmark 2 completed!")